In [ ]:
%pip install numpy matplotlib qiskit qiskit-aer scipy --quiet

In [ ]:
from __future__ import annotations
import math
from dataclasses import dataclass, field
from typing import Sequence
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import MCXGate, RYGate
from qiskit.quantum_info import Statevector

In [ ]:
#Quantum Analog of classical RAM pebble game

@dataclass
class step:
    node: int
    action: str

def pebble_schedule(dag_length:int, budget:int | None = None) -> list[step]:
    if budget is None or budget >= dag_length:
        sched: list[step] = []  #Forward pass
        for i in range(dag_length):
            sched.append(step(i,"GO"))
        for i in reversed(range(dag_length-1)):
            sched.append(step(i,"NO-GO"))
        return sched

    def _recurse(low:int, high:int, budg:int) -> list[step]:
        if high-low <= 1:
            return [step(low,"GO")]
        if budg <= 1:
            seq = []
            for i in range(low, hi):
                seq.append(step(i,"GO"))
            return seq
        mid = (low + high)//2   #Pebble entry graph
        left = _recurse(low, mid, budg-1)
        right = _recurse(mid, high, budg-1)
        left_un = [step(s.node,"NO-GO") for s in reversed(left) if s.action == "GO"]

        return left + right + left_un

    return _recurse(0, dag_length, budg)

def quantum_pebble(dag_length:int, class_mem:bool = True) -> list[step]:  #Quantum version with mid-circuit measurement
    sched: list[step] = []
    for i in range(dag_length):
        sched.append(step(i,"GO"))
    if class_mem:
        for i in range(dag_length - 1):
            sched.append(step(i,"Measure"))
    else:
        for i in reversed(range(dag_length - 1)):
            sched.append(step(i,"NO-GO"))
    return sched

In [ ]:
#Virtual Memory Sim

@dataclass
class VirtualAddress:
    segment_index: int
    intra_segment_offset: int
    k_bits: int
    m_bits: int

def quantum_vm(addr:int, k_bits:int, m_bits: int) -> VirtualAddress:
    total = k_bits + m_bits
    assert 0 <= addr < (1 << total), "Address out of range!"
    segment = addr >> m_bits
    offset = addr & ((1 << m_bits)-1)
    return VirtualAddress(segment, offset, k_bits, m_bits)

In [ ]:
#Grover-Rudolph State Prep

def _reverse_bit(n:int) -> np.ndarray:
    perm = np.zeros(2**n, dtype=int)
    for i in range(2**n):
        r = 0
        for k in range(n):
            if (i >> k) & 1:
                r |= 1 << (n-1-k)
        perm[i] = r
    return perm

def gr_angles(probs:np.ndarray) -> list[list[float]]:  #Does the computation based on conditional rotational angles
    n = int(math.log2(len(probs)))
    p = np.asarray(probs, dtype=float)
    assert np.isclose(p.sum(),1.0)
    angles: list[list[float]] = []

    for i in range(n):
        level: list[float] = []
        for prefix in range(2**i):
            mask_low = prefix << (n-i)
            mask_high = (prefix+1) << (n-i)
            total = p[mask_low:mask_high].sum()
            if total < 1e-15:               #Negligible values
                level.append(0.0)
                continue
            half = (mask_low + mask_high)//2   #Significant part of the tree
            right = p[half:mask_high].sum()
            ratio = np.clip(right/total, 0.0, 1.0)
            level.append(2.0*np.arcsin(np.sqrt(ratio)))
        angles.append(level)

    return angles

def gr_circuit(probs:np.ndarray) -> QuantumCircuit:
    n = int(math.log2(len(probs)))

    qr = QuantumRegister(n, name='q')
    qc = QuantumCircuit(qr, name="GR_CIRCUIT")
    angles = gr_angles(probs)

    for i, level in enumerate(angles):
        for prefix, theta in enumerate(level):
            if abs(theta) < 1e-12:        #Controls the first i qubits
                continue
            bits = [(prefix >> (i-1-k)) & 1 for k in range(i)]
            for k, b in enumerate(bits):  #Activation if the state is |0>
                if b == 0:
                    qc.x(qr[k])
            if i == 0:
                qc.ry(theta,qr[i])
            else:
                qc.append(RYGate(theta).control(i),[qr[k] for k in range(i)] + [qr[i]])   #The rotational gates
            for k, b in enumerate(bits):
                if b == 0:
                    qc.x(qr[k])
    return qc

In [ ]:
#The Kerenidis-Prakash tree

@dataclass
class KPTree:
    n_bits: int
    sqrd_norms: list[np.ndarray] = field(default_factory=list)
    signs:np.ndarray | None = None

    @classmethod
    def from_vector(cls, x:np.ndarray) -> "KPTree":
        x = np.asarray(x, dtype=float)
        n = int(math.log2(len(x)))
        signs = np.sign(x)
        signs[signs == 0] = 1.0
        sqrd = [x.astype(float)**2]
        curr = sqrd[0].copy()
        for _ in range(n):
            curr = curr.reshape(-1, 2).sum(axis=1)
            sqrd.append(curr.copy())
        sqrd = list(reversed(sqrd))

        return cls(n_bits=n, sqrd_norms=sqrd, signs=signs)

    def norm(self) -> float:
        return math.sqrt(self.sqrd_norms[0][0])

    def cond_prob(self, level:int, prefix:int) -> float:
        parent = self.sqrd_norms[level][prefix]
        if parent < 1e-18:
            return 0.0
        right_child = self.sqrd_norms[level+1][2*prefix+1]

        return float(right_child/parent)


def kp_circuit(tree:KPTree) -> QuantumCircuit:
    n = tree.n_bits

    qr = QuantumRegister(n, name='q')
    qc = QuantumCircuit(qr, name="KP_Circuit")

    for i in range(n):
        for prefix in range(2**i):
            ratio = tree.cond_prob(i,prefix)
            if ratio < 1e-15 and ratio > -1e-15:
                theta = 0.0
            else:
                theta = 2.0*np.arcsin(math.sqrt(max(0.0,ratio)))
            if abs(theta) < 1e-12:
                continue
            bits = [(prefix >> (i-1-k)) & 1 for k in range(i)]
            for k, b in enumerate(bits):
                if b == 0:
                    qc.x(qr[k])
            if i == 0:
                qc.ry(theta, qr[i])
            else:
                qc.append(RYGate(theta).control(i), [qr[k] for k in range(i)] + [qr[i]])
            for k, b in enumerate(bits):
                if b == 0:
                    qc.x(qr[k])

    for i, s in enumerate(tree.signs):        #Phase loading from the QRAM
        if s < 0:
            bits = [(i >> (n-1-k)) & 1 for k in range(n)]
            for k, b in enumerate(bits):
                if b == 0:
                    qc.x(qr[k])
            qc.append(MCXGate(n-1).control(), [qr[k] for k in range(n)])
            if False else qc.append(MCXGate(n-1), [qr[k] for k in range(n)])
            for k, b in enumerate(bits):
                if b == 0:
                    qc.x(qr[k])

    return qc

In [ ]:
#Sparse state prep

def sparse_circuit(amp:dict[int,complex], n_bits:int) -> QuantumCircuit:
    qr = QuantumRegister(n_bits, name='q')
    qc = QuantumCircuit(qr, name="SparseStatePrep")

    support = sorted(amp.keys())
    if not support:
        return qc
    amps = np.array([amp[s] for s in support])
    total_norm_sq = float(np.vdot(amps, amps).real)
    if not np.isclose(total_norm_sq, 1.0):
        amps = amps/np.sqrt(total_norm_sq)

    for idx, (basis, a) in enumerate(zip(support,amps)):
        bits = [(basis >> (n_bits-1-k)) & 1 for k in range(n_bits)]
        if idx == 0:
            for k, b in enumerate(bits):
                if b == 1:
                    qc.x(qr[k])
            continue
        theta = 2.0*np.arccos(min(1.0, abs(a)))
        for k, b in enumerate(bits):
            if b == 0:
                qc.x(qr[k])
        qc.append(RYGate(theta).control(n_bits-1), [qr[k] for k in range(1, n_bits)] + [qr[0]])

        for k, b in enumerate(bits):
            if b == 0:
                qc.x(qr[k])
    return qc

In [ ]:
#Small demo

def _demo_layer3() -> None:
    print("Layer 3")

    sched_class = pebble_schedule(8, budget=3)  #Demo values
    sched_quant = quantum_pebble(8, class_mem=True)
    print(f"Classical schedule: {len(sched_class)} steps")
    print(f"Quanutm schedule: {len(sched_quant)} steps")

    addr = 0b10110110  #Demo 8-bit address for virtual memory
    va = quantum_vm(addr, k_bits=3, m_bits=5)
    print(f"Virtual memory address 0b{addr:08b}->segment {va.segment_index}, " f"offset {va.intra_segment_offset} " f"(k={va.k_bits}, m={va.m_bits})")

    x = np.arange(8)-3.5  #GR state demo
    p = np.exp(-x**2/2.0)
    p /= p.sum()
    perm = _reverse_bit(3)
    qc_gr = gr_circuit(p[perm])
    sv_gr = Statevector.from_instruction(qc_gr)
    probs_gr = np.abs(sv_gr.data)**2
    err = np.max(np.abs(probs_gr - p))
    print(f"GR State: max error = " f"{err:.2e}")

    x_vec = np.array([0.3, 0.5, -0.4, 0.2, 0.1, 0.6, -0.3, 0.1])  #KP tree demo
    x_vec /= np.linalg.norm(x_vec)
    tree = KPTree.from_vector(x_vec[perm])
    qc_kp = kp_circuit(tree)
    sv_kp = Statevector.from_instruction(qc_kp)
    probs_kp = np.abs(sv_kp.data)**2
    err_kp = np.max(np.abs(probs_kp - x_vec**2))
    print(f"KP tree: max error = " f"{err_kp:.2e}")

    sparse_amps = {0b000: 1.0/math.sqrt(3), 0b011: 1.0/math.sqrt(3), 0b111: 1.0/math.sqrt(3)}   #Sparse state demo
    qc_sp = sparse_circuit(sparse_amps, n_bits=3)
    sv_sp = Statevector.from_instruction(qc_sp)
    probs_sp = np.abs(sv_sp.data)**2
    wght = probs_sp[0b000] + probs_sp[0b011] + probs_sp[0b111]
    print(f"Sparse state: {wght:.4f}")

In [ ]:
if __name__ == "__main__":
    _demo_layer3()